## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🏦 Azure AI Agent Basic Example 💼

This notebook demonstrates basic usage of `Agent` with `FoundryChatClient` for creating agents powered by Azure AI Foundry.

## Features Covered:
- Setting up `FoundryChatClient` and `Agent`
- Creating an advisor workflow with function tools
- Using function tools for banking operations
- Streaming and non-streaming responses

### ⚠️ Important Financial Disclaimer ⚠️
> **The financial information provided by this notebook is for general educational and demonstration purposes only and is not intended as financial, investment, legal, or tax advice.** Always consult with qualified financial advisors before making any financial decisions.

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Foundry Project**: Access to a Foundry project with a deployed model
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with connection details:
   - `AI_FOUNDRY_PROJECT_ENDPOINT` — your Foundry project endpoint
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the model deployment name (e.g., `gpt-4o`)
4. **Dependencies**: Required agent-framework packages installed (including `agent-framework-foundry`)

If you need to use a different tenant, specify the tenant ID:
```bash
az login --tenant <tenant-id>
```

## Import Required Libraries

First, let's import the necessary libraries. We use `Agent` with `FoundryChatClient` to create an agent powered by Azure AI Foundry:

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import sys
from importlib.metadata import version
from random import randint, uniform
from typing import Annotated

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from pydantic import Field

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print({p: version(p) for p in ("agent-framework-core", "agent-framework-foundry", "agent-framework-openai", "azure-ai-projects")})

## Initial Setup

Load environment variables from the `.env` file for Azure AI Project configuration:

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

# Resolve the root from either the notebook directory or repository directory.
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.in").is_file())
load_dotenv(repo_root / ".env", override=False)
endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), "Select the repository .venv kernel."
print("Project endpoint and model: configured (values hidden)")

## Define Function Tools 🏦

Function tools allow the agent to call specific functions to gather information or perform operations. Here we define functions for account balance and loan rate inquiries:

In [ ]:
def get_account_balance(
    account_id: Annotated[str, Field(description="The customer account ID to check balance for.")],
) -> str:
    """Get the current balance for a customer account."""
    # Simulated account balances for demo
    balances = {
        "checking": round(uniform(1000, 15000), 2),
        "savings": round(uniform(5000, 50000), 2),
        "investment": round(uniform(10000, 100000), 2)
    }
    account_type = ["checking", "savings", "investment"][randint(0, 2)]
    return f"Account {account_id} ({account_type}): Current balance is ${balances[account_type]:,.2f}"


def get_loan_rates(
    loan_type: Annotated[str, Field(description="Type of loan: mortgage, auto, personal, or business")],
) -> str:
    """Get current interest rates for different loan types."""
    rates = {
        "mortgage": {"rate": 6.5, "term": "30 years", "min_credit": 620},
        "auto": {"rate": 7.2, "term": "5 years", "min_credit": 600},
        "personal": {"rate": 10.5, "term": "3 years", "min_credit": 650},
        "business": {"rate": 8.0, "term": "10 years", "min_credit": 680}
    }
    loan_type = loan_type.lower()
    if loan_type in rates:
        r = rates[loan_type]
        return f"Current {loan_type} loan rates: {r['rate']}% APR, {r['term']} term, minimum credit score: {r['min_credit']}"
    return f"Unknown loan type: {loan_type}. Available types: mortgage, auto, personal, business"


# Financial disclaimer constant
FINANCIAL_DISCLAIMER = """
⚠️ DISCLAIMER: This information is for educational purposes only and does not constitute 
financial advice. Please consult with a qualified financial advisor for personalized guidance.
"""

## Non-Streaming Response Example 💰

In this example, we create a `FoundryChatClient` and pass it to an `Agent` to get a complete response at once (non-streaming):

- `FoundryChatClient(project_endpoint=..., model=..., credential=...)` connects to Azure AI Foundry
- `Agent(client=..., name=..., instructions=..., tools=[...])` creates the agent with function tools

In [ ]:
async def non_streaming_example() -> None:
    """Get a complete response using application-owned agent configuration."""
    print("=== 🏦 Non-streaming Response Example ===")
    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="FinancialAdvisorAgent",
                instructions="""You are a helpful Financial Services Advisor for a retail bank.
                Use the demo tools to check account balances and inquire about loan rates.
                Always be professional and provide clear, helpful information.""",
                tools=[get_account_balance, get_loan_rates],
            )
            query = "What are the current mortgage loan rates?"
            print(f"🤔 Customer: {query}")
            async with asyncio.timeout(90):
                result = await agent.run(query)
            assert result.text, "The service returned no answer."
            print(f"🏦 Advisor: {result.text}")
            print(FINANCIAL_DISCLAIMER)
        finally:
            await client.client.close()
            await client.project_client.close()

## Streaming Response Example 📊

In this example, we'll demonstrate streaming responses by calling `agent.run(query, stream=True)` and iterating updates as they arrive:

In [ ]:
async def streaming_example() -> None:
    """Stream response updates, with a bounded service call."""
    print("=== 📊 Streaming Response Example ===")
    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="FinancialAdvisorAgent",
                instructions="""You are a helpful Financial Services Advisor for a retail bank.
                Use the demo tools to check account balances and inquire about loan rates.
                Always be professional and provide clear, helpful information.""",
                tools=[get_account_balance, get_loan_rates],
            )
            query = "Can you check the balance for account ACC-12345 and also tell me about auto loan rates?"
            print(f"🤔 Customer: {query}\n🏦 Advisor: ", end="", flush=True)
            text = ""
            async with asyncio.timeout(90):
                async for chunk in agent.run(query, stream=True):
                    if chunk.text:
                        text += chunk.text
                        print(chunk.text, end="", flush=True)
            assert text, "The service returned no streamed answer."
            print("\n" + FINANCIAL_DISCLAIMER)
        finally:
            await client.client.close()
            await client.project_client.close()

## Main Execution Function

This function orchestrates the execution of both examples:

In [ ]:
async def main() -> None:
    print("=== 🏦 Azure AI Agent Example ===\n")
    failures = []
    # Validate both independent service paths even if one fails; never report a swallowed failure.
    for example in (non_streaming_example, streaming_example):
        try:
            await example()
        except Exception as exc:
            print(f"{example.__name__}: {type(exc).__name__}: {exc}")
            failures.append(exc)
    if failures:
        raise ExceptionGroup("Financial advisor live calls failed", failures)

## Run the Examples 🚀

Execute the main function to run both streaming and non-streaming Financial Advisor examples:

In [ ]:
# Run the main function
await main()

## Key Takeaways 📚

1. **Application-owned configuration**: use `Agent(client=FoundryChatClient(...), tools=[...])`.
2. **Existing service-managed agents**: use `FoundryAgent(agent_name=...)`; this API has **not** been removed. See notebook 3.
3. **Function tools**: plain annotated functions are supported; the banking data here is simulated, but model calls are live.
4. **Responses**: `await agent.run(...)` returns `.text`; `agent.run(..., stream=True)` streams updates.
5. **Lifecycle**: close the underlying OpenAI and project clients. No persistent agent is created or deleted by this example.
6. **Configuration**: explicitly load environment settings; legacy repository names are mapped to current constructor arguments without modifying the environment file.

### Pinned migration references

Validated against `agent-framework-core==1.17.0`, with the provider versions printed by the setup cell. Normal execution never installs or upgrades packages.
- [Python significant changes](https://learn.microsoft.com/en-us/agent-framework/support/upgrade/python-2026-significant-changes)
- [Core 1.17.0 release](https://pypi.org/project/agent-framework-core/1.17.0/)

Cloud operations are bounded to 90 seconds. Errors propagate, including failures in either streaming or non-streaming validation.